# DMF — Calibración SVG (Python)
Réplica del ejemplo Python del showcase `Factor_DMF_Showcase.hcalc`.
Modifica los parámetros para calibrar colores, grosores, rangos, etc.

In [ ]:
import math
from IPython.display import SVG, display, HTML

# === PARAMETROS DE CALIBRACION ===
W, H = 650, 400           # Ancho y alto del SVG
ml, mr, mt, mb = 55, 25, 30, 45  # Margenes: left, right, top, bottom
xmin, xmax = 0, 3         # Rango eje X (r = freq/freq_nat)
ymin, ymax = 0, 4.5       # Rango eje Y (D = factor amplificacion)
n_points = 300             # Numero de puntos por curva

# Valores de amortiguamiento (xi)
xi_vals = [0.125, 0.2, 0.25, 0.5, 0.707, 1.0, 2.0]

# Colores para cada curva
colors = ['#E53935', '#7CB342', '#1E88E5', '#FFB300', '#C2185B', '#00C853', '#7B1FA2']

# Etiquetas
titulo = 'Factor de Magnificación Dinámica (DMF)'
xlabel = 'r = ω/ωₙ'
ylabel = 'D(r,ξ)'

# Estilo de lineas
stroke_width_main = '2'       # Grosor lineas principales (xi < 0.707)
stroke_width_secondary = '1.5' # Grosor lineas secundarias
dash_threshold = 3            # Indice a partir del cual usar linea punteada (0-based)
dash_pattern = '8,4'          # Patron de guiones

# Fuentes
font_family = 'Segoe UI, sans-serif'
font_size_title = 14
font_size_axis = 11
font_size_legend = 10

print(f'Canvas: {W}x{H}  |  Plot area: {W-ml-mr}x{H-mt-mb}  |  Curvas: {len(xi_vals)}')

In [ ]:
# === FUNCIONES ===
pw, ph = W - ml - mr, H - mt - mb

def sx(v):
    """Escalar X a coordenadas SVG"""
    return ml + (v - xmin) / (xmax - xmin) * pw

def sy(v):
    """Escalar Y a coordenadas SVG (invertido)"""
    return mt + ph - (v - ymin) / (ymax - ymin) * ph

def D(r, xi):
    """Factor de Magnificación Dinámica"""
    denom = (1 - r**2)**2 + (2*xi*r)**2
    return min(1/math.sqrt(denom), ymax) if denom > 1e-12 else ymax

# === GENERAR SVG ===
o = []
o.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" style="font-family:{font_family};background:white">')

# Grid vertical
for i in range(int(xmax) + 1):
    x = sx(i)
    o.append(f'<line x1="{x:.1f}" y1="{mt}" x2="{x:.1f}" y2="{mt+ph}" stroke="#e8e8e8"/>')
    o.append(f'<text x="{x:.1f}" y="{mt+ph+16}" text-anchor="middle" font-size="{font_size_axis}" fill="#555">{i}</text>')

# Grid horizontal
for j in range(int(ymax) + 1):
    y = sy(j)
    o.append(f'<line x1="{ml}" y1="{y:.1f}" x2="{ml+pw}" y2="{y:.1f}" stroke="#e8e8e8"/>')
    o.append(f'<text x="{ml-8}" y="{y+4:.1f}" text-anchor="end" font-size="{font_size_axis}" fill="#555">{j}</text>')

# Marco
o.append(f'<rect x="{ml}" y="{mt}" width="{pw}" height="{ph}" fill="none" stroke="#999"/>')

# Titulo y etiquetas de eje
o.append(f'<text x="{W/2}" y="20" text-anchor="middle" font-size="{font_size_title}" font-weight="bold">{titulo}</text>')
o.append(f'<text x="{ml+pw/2}" y="{H-5}" text-anchor="middle" font-size="{font_size_axis}" fill="#555">{xlabel}</text>')
o.append(f'<text x="12" y="{mt+ph/2}" text-anchor="middle" font-size="{font_size_axis}" fill="#555" transform="rotate(-90,12,{mt+ph/2})">{ylabel}</text>')

# Curvas DMF
r_vals = [xmin + 0.01 + i * (xmax - 0.01) / (n_points - 1) for i in range(n_points)]
for k, xi in enumerate(xi_vals):
    pts = ' '.join(f'{sx(r):.1f},{sy(D(r,xi)):.1f}' for r in r_vals)
    lw = stroke_width_main if k <= dash_threshold else stroke_width_secondary
    dash = f' stroke-dasharray="{dash_pattern}"' if k > dash_threshold else ''
    o.append(f'<polyline points="{pts}" fill="none" stroke="{colors[k]}" stroke-width="{lw}"{dash}/>')

# Leyenda
lx = ml + pw - 120
ly_s = mt + 12
n = len(xi_vals)
o.append(f'<rect x="{lx-5}" y="{ly_s-6}" width="125" height="{n*16+8}" rx="3" fill="white" fill-opacity="0.9" stroke="#ddd"/>')
for k in range(n):
    yy = ly_s + 8 + k * 16
    dash = f' stroke-dasharray="6,3"' if k > dash_threshold else ''
    o.append(f'<line x1="{lx}" y1="{yy}" x2="{lx+22}" y2="{yy}" stroke="{colors[k]}" stroke-width="2"{dash}/>')
    o.append(f'<text x="{lx+27}" y="{yy+4}" font-size="{font_size_legend}">ξ = {xi_vals[k]}</text>')

o.append('</svg>')
svg_text = '\n'.join(o)

# Mostrar SVG
display(SVG(svg_text))
print(f'SVG generado: {len(svg_text)} bytes, {len(o)} elementos')

## Tabla de valores D(r=1,ξ)
Verificación numérica de los valores en resonancia (r=1)

In [ ]:
print(f'{"xi":>8s} | {"D(r=1)":>8s} | {"D_max":>8s} | {"r_max":>8s}')
print('-' * 42)
for xi in xi_vals:
    d_res = 1 / (2 * xi)  # D en resonancia (r=1)
    # D maximo se da en r = sqrt(1 - 2*xi^2) si xi < 1/sqrt(2)
    if xi < 1/math.sqrt(2):
        r_peak = math.sqrt(1 - 2*xi**2)
        d_max = D(r_peak, xi)
    else:
        r_peak = 0.0
        d_max = 1.0  # D(0) = 1 siempre
    print(f'{xi:8.3f} | {d_res:8.4f} | {d_max:8.4f} | {r_peak:8.4f}')

## Exportar SVG a string (para pegar en Hekatan)
Copia el output de la celda siguiente para pegarlo en un bloque `@{python}` de Hekatan.

In [ ]:
# Imprimir SVG como lo espera Hekatan (print directo)
print(svg_text)